# SystemOne Guard versus pretrained SmolLM2 instruction models
Run on internet-connected Google Colab with CPU runtime. Two genuine instruction-tuned HF models. The same example IDs, same CPU threads, and measured end-to-end latency.

## Upload corrected ZIP
Upload `systemone-guard-pretrained-compare.zip`. The pretrained HF weights are downloaded by the benchmark, not included in the ZIP.

In [ ]:
import os
import pathlib
import zipfile

from google.colab import files

uploaded = files.upload()
with zipfile.ZipFile(next(iter(uploaded))) as z:
    z.extractall("/content")
root = pathlib.Path("/content/systemone-guard-pretrained-compare")
assert root.is_dir()
os.chdir(root)
print(root)

In [ ]:
!pip -q install -e ".[llm,dev,notebook]"

## Matched benchmark
Start with COUNT=20. Change to 100 for reported subset, or 0 for the entire split. Both run on CPU for apples-to-apples timing, limited to 2 CPU threads.

In [ ]:
COUNT = 20
SPLIT = "test"
!python -m systemone_guard.benchmarking.pretrained --split {SPLIT} --count {COUNT} --seed 42 --threads 2 --device cpu --out experiments/pretrained_{SPLIT}_{COUNT}.json

In [ ]:
!python scripts/plots.py pretrained --input experiments/pretrained_{SPLIT}_{COUNT}.json --out figures/pretrained_{SPLIT}_{COUNT}

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"figures/pretrained_{SPLIT}_{COUNT}/accuracy_latency.png"))

In [ ]:
import json

import pandas as pd

res = json.loads(
    pathlib.Path(f"experiments/pretrained_{SPLIT}_{COUNT}.json").read_text()
)
display(
    pd.DataFrame(res["summaries"])[
        [
            "method",
            "n",
            "accuracy",
            "macro_f1",
            "invalid_rate",
            "median_latency_ms",
            "p95_latency_ms",
        ]
    ]
)

## Inspect decisions and actual outputs
A pretrained LLM may return invalid text rather than a typed choice. Those are incorrect and included in the invalid-rate metric.

In [ ]:
df = pd.DataFrame(res["records"])
display(df[["id", "method", "gold", "prediction", "latency_ms"]].head(20))
base = res["summaries"][0]["median_latency_ms"]
for row in res["summaries"][1:]:
    print(
        row["method"],
        "LLM/Guard latency ratio",
        round(row["median_latency_ms"] / base, 2),
    )

## For external validity
Repeat with SPLIT="challenge" and separately review the documented 24 manual probes. Do not claim the synthetic security results generalize to novel prompt-injection attacks.

In [ ]:
!python -m pytest -q tests/test_pretrained_comparator.py